**Notebook overview**
- Purpose: Fetch and process Guardian articles for a target date window.
- Produces: raw JSON in `data/raw/` and cleaned CSV(s) in `data/processed/` using the schema: `date`, `news source`, `title`, `link`.
- Notes: Processing includes keyword filtering and duplicate removal.



In [1]:
# Purpose: define a function to query The Guardian and return articles matching conflict signals
from datetime import datetime, timedelta
from typing import Optional

import pandas as pd
import requests


def get_guardian_headlines(start_date_str, total_days, api_key, country="Ukraine", return_raw=False, page_size=200, max_pages=None):
    """Fetch conflict-related articles from The Guardian Content API."""
    missile_terms = "missile OR cruise missile OR rocket OR barrage OR salvo OR attack OR attacked OR attacks OR strike OR struck OR strikes OR striked OR explosion OR exploded OR explodes OR explode OR blast OR blasted OR bomb OR bombed OR bombing OR killed OR killing OR injured OR injury OR hit OR hits OR shelling OR bombardment"
    explosion_terms = "explosion OR blast OR strike OR attack OR hit OR bombardment OR shelling"
    drone_terms = "drone OR kamikaze OR shahed OR uav"
    impact_terms = "blackout OR power outage OR emergency shutdown OR kill OR casualty OR injury OR destruction OR wildfire OR forest fire OR forestfire OR fire OR brush fire OR bushfire OR blaze OR inferno OR arson OR flames OR burning OR smoke OR spread OR spreaded OR wind OR burn OR burned"

    # Combine terms into the query string used by the Guardian API
    optional_signals = f"({missile_terms} OR {drone_terms} OR {impact_terms})"
    conflict_signals = f"({explosion_terms}) AND ({optional_signals} OR {explosion_terms})"
    query_string = f"{country} AND {conflict_signals}" if country else conflict_signals

    start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_dt = start_dt + timedelta(days=total_days - 1)
    end_date_str = end_dt.strftime("%Y-%m-%d")

    print("=" * 60)
    print(f"Query target: {country if country else 'Global'}")
    print(f"Timeframe: {start_date_str} to {end_date_str} ({total_days} day(s))")
    print(f"Paging: page_size={page_size}, max_pages={'all' if max_pages is None else max_pages}")
    print("=" * 60)

    url = "https://content.guardianapis.com/search"
    params = {
        "q": query_string,
        "from-date": start_date_str,
        "to-date": end_date_str,
        "api-key": api_key,
        "page-size": page_size,
        "order-by": "oldest",
        "show-fields": "headline,trailText",
    }

    all_rows = []
    raw_articles = []
    page = 1

    while True:
        if max_pages is not None and page > max_pages:
            break

        params["page"] = page
        try:
            response = requests.get(url, params=params, timeout=20)
            response.raise_for_status()
            payload = response.json()
            data = payload.get("response", {})

            if data.get("status") != "ok":
                print("Guardian API returned non-ok status:", data.get("status"))
                break

            results = data.get("results", [])
            if not results:
                break

            for article in results:
                fields = article.get("fields", {}) or {}
                all_rows.append(
                    {
                        "source": "The Guardian",
                        "country": country if country else "Global",
                        "date": article.get("webPublicationDate", "")[:10],
                        "published_at_utc": article.get("webPublicationDate", ""),
                        "headline": fields.get("headline") or article.get("webTitle", ""),
                        "snippet": fields.get("trailText", ""),
                        "section": article.get("sectionName", ""),
                        "url": article.get("webUrl", ""),
                        "api_id": article.get("id", ""),
                    }
                )
                raw_articles.append(
                    {
                        "query": query_string,
                        "page": page,
                        "fetched_at_utc": datetime.utcnow().isoformat() + "Z",
                        "article": article,
                    }
                )

            pages = data.get("pages", 1)
            if page >= pages:
                break
            page += 1

        except requests.RequestException as exc:
            print("Guardian request failed:", exc)
            break

    if not all_rows:
        print("No articles matched this query/time window.")
        if return_raw:
            return pd.DataFrame(), raw_articles
        return pd.DataFrame()

    df_news = pd.DataFrame(all_rows)
    df_news["date"] = pd.to_datetime(df_news["date"], errors="coerce")
    df_news = df_news.drop_duplicates(subset=["url"]).sort_values("published_at_utc").reset_index(drop=True)
    print(f"Collected {len(df_news)} articles from The Guardian.")

    if return_raw:
        return df_news, raw_articles
    return df_news


In [12]:
# --- Configuration ---
GUARDIAN_API_KEY = "8fa62376-0756-4583-bbf4-a732a4c28c54"  # Replace with your actual key
START = "2026-01-01"
DAYS_TO_FETCH = 180  # Oct 10 to Oct 14 in a single query window
TARGET_COUNTRY = ""
PAGE_SIZE = 200
MAX_PAGES = None  # set a number like 5 if you want to cap pagination

# --- Run the Pipeline ---
df_guardian_headlines, raw_guardian_articles = get_guardian_headlines(
    start_date_str=START,
    total_days=DAYS_TO_FETCH,
    api_key=GUARDIAN_API_KEY,
    country=TARGET_COUNTRY,
    return_raw=True,
    page_size=PAGE_SIZE,
    max_pages=MAX_PAGES,
)

# --- Save raw data only ---
from pathlib import Path
import json

RAW_DIR = Path("./data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSON = RAW_DIR / f"guardian_raw_{START.replace('-', '')}_{DAYS_TO_FETCH}d.json"

with open(RAW_JSON, "w", encoding="utf-8") as f:
    json.dump(
        {
            "source": "The Guardian Open Platform",
            "query_start_date": START,
            "query_total_days": DAYS_TO_FETCH,
            "country": TARGET_COUNTRY,
            "page_size": PAGE_SIZE,
            "max_pages": "all" if MAX_PAGES is None else MAX_PAGES,
            "raw_article_count": len(raw_guardian_articles),
            "articles": raw_guardian_articles,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

# --- Inspect results in memory only ---
if df_guardian_headlines.empty:
    print("No data returned. Check API key, query strictness, or widen date range.")
else:
    print(df_guardian_headlines.head(10))
    print("\nColumns:", list(df_guardian_headlines.columns))
    print("\nDate range:", df_guardian_headlines["date"].min(), "to", df_guardian_headlines["date"].max())
    print("\nTotal rows:", len(df_guardian_headlines))

print("Saved raw JSON to:", RAW_JSON.resolve())
print("Metadata written: source, query_start_date, query_total_days, page_size, max_pages")

Query target: Global
Timeframe: 2026-01-01 to 2026-06-29 (180 day(s))
Paging: page_size=200, max_pages=all


C:\Users\AFT\AppData\Local\Temp\ipykernel_30100\1125162765.py:84: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at_utc": datetime.utcnow().isoformat() + "Z",


Collected 3035 articles from The Guardian.
         source country       date      published_at_utc  \
0  The Guardian  Global 2026-01-01  2026-01-01T04:48:32Z   
1  The Guardian  Global 2026-01-01  2026-01-01T05:00:29Z   
2  The Guardian  Global 2026-01-01  2026-01-01T06:00:30Z   
3  The Guardian  Global 2026-01-01  2026-01-01T06:00:33Z   
4  The Guardian  Global 2026-01-01  2026-01-01T09:00:34Z   
5  The Guardian  Global 2026-01-01  2026-01-01T13:00:40Z   
6  The Guardian  Global 2026-01-01  2026-01-01T13:01:39Z   
7  The Guardian  Global 2026-01-01  2026-01-01T18:43:49Z   
8  The Guardian  Global 2026-01-01  2026-01-01T19:25:35Z   
9  The Guardian  Global 2026-01-01  2026-01-01T20:52:15Z   

                                            headline  \
0  Australian beef industry ‘extremely disappoint...   
1  Grief, fear and fury: the Israeli and Palestin...   
2  Brawls, blackmail and Judi Dench: 75 staggerin...   
3  When racists shout ‘Go home’, and you come fro...   
4  How this stra

In [13]:
# --- Process raw Guardian JSON -> standardized CSV(s) ---
import json
import re
from pathlib import Path

import pandas as pd

ANALYSIS_KEYWORDS = [
    "missile",
    "explosion",
    "drone",
    "strike",
    "attack",
    "shelling",
    "bombardment",
    "blast",
    "blackout",
    "power outage",
    "infrastructure",
    "energy",
    "missile",
    "strike",
    "explosion",
    "drone",
    "barrage",
    "blast",
    "attack",
    "bombardment",
    "shelling",
    "kamikaze",
    "shahed",
    "blackout",
    "power outage",
    "energy",
    "electricity",
    "grid",
    "infrastructure",
    "attack",
    "attacked",
    "attacks",
    "strike",
    "struck",
    "strikes",
    "striked",
    "explosion",
    "exploded",
    "explodes",
    "explode",
    "blast",
    "blasted",
    "bomb",
    "bombed",
    "bombing",
    "killed",
    "killing",
    "injured",
    "injury",
    "hit",
    "hits",
    "shelling",
    "bombardment",
    "burning",
    "burned",
    "fire",
    "wildfire",
    "forest fire",
    "forestfire",
    "fire",
    "brush fire",
    "bushfire",
    "blaze",
    "inferno",
    "arson",
    "flames",
    "burning",
    "smoke",
    "spread",
    "spreaded",
    "wind",
    "burn",
    "burned",
]

RAW_DIR = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


def merge_headline_and_trailtext(headline, trail_text):
    headline_text = str(headline or "").strip()
    trail_text = re.sub(r"<[^>]+>", " ", str(trail_text or ""))
    trail_text = re.sub(r"\s+", " ", trail_text).strip()
    if headline_text and trail_text and trail_text.lower() != headline_text.lower():
        return f"{headline_text} - {trail_text}"
    return headline_text or trail_text


for raw_file in sorted(RAW_DIR.glob("guardian_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to read {raw_file}: {exc}")
        continue

    articles = obj.get("articles", [])
    rows = []
    for rec in articles:
        art = rec.get("article", {})
        fields = art.get("fields", {}) or {}
        headline = fields.get("headline") or art.get("webTitle", "")
        trail_text = fields.get("trailText", "")
        title = merge_headline_and_trailtext(headline, trail_text)
        link = art.get("webUrl", "")
        if not title or not link:
            continue

        snippet = trail_text
        if not matches_keywords(f"{headline} {snippet}"):
            continue

        rows.append(
            {
                "date": parse_datetime(art.get("webPublicationDate", "")),
                "news source": "Guardian",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching articles found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed Guardian data to: {processed_path} ({len(df)} rows)")


Saved processed Guardian data to: data\processed\guardian_20190101_360d.csv (550 rows)
Saved processed Guardian data to: data\processed\guardian_20210701_90d.csv (22 rows)
Saved processed Guardian data to: data\processed\guardian_20221001_1280d.csv (2928 rows)
Saved processed Guardian data to: data\processed\guardian_20221001_180d.csv (541 rows)
Saved processed Guardian data to: data\processed\guardian_20260101_180d.csv (2130 rows)


**What this notebook does**
- Queries The Guardian Content API for the selected date window and saves the raw API payload under `data/raw/`.
- Re-processes each saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read the nested `article` records from the saved raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse `webPublicationDate` into a full datetime and keep the time from the source.
- Set the news source label to `Guardian`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging All`

In [14]:
# Incremental merge for Guardian: keep history in guardian_all.csv and append newcomers
from pathlib import Path
import pandas as pd

PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "guardian_all.csv"

guardian_files = sorted(
    f for f in PROC_DIR.glob("guardian_*.csv")
    if f.name.lower() != "guardian_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in guardian_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No Guardian data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental Guardian master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None

Loaded existing master guardian_all.csv: 3500 rows
Loaded guardian_20190101_360d.csv: 550 rows
Loaded guardian_20210701_90d.csv: 22 rows
Loaded guardian_20221001_1280d.csv: 2928 rows
Loaded guardian_20221001_180d.csv: 541 rows
Loaded guardian_20260101_180d.csv: 2130 rows
Saved incremental Guardian master: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\News\data\processed\guardian_all.csv (5439 rows)


,date,news source,title,link
0,2022-10-04 09:36:55+00:00,Guardian,Ukraine: at least 18 people working for occupi...,https://www.theguardian.com/world/2022/oct/04/...
1,2022-10-04 12:02:52+00:00,Guardian,Shell chief: governments may need to tax energ...,https://www.theguardian.com/business/2022/oct/...
2,2022-10-04 19:58:23+00:00,Guardian,"Morning mail: security firm data breach, UK ca...",https://www.theguardian.com/australia-news/202...
3,2022-10-05 16:58:17+00:00,Guardian,Nicole Mann becomes first Native American woma...,https://www.theguardian.com/science/2022/oct/0...
4,2022-10-05 21:00:18+00:00,Guardian,Climate crisis made summer drought 20 times mo...,https://www.theguardian.com/environment/2022/o...
